## Set up Data

#### Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Unzip the Images

In [ ]:
import zipfile, os

CHALLENGE_DIR = "/content/drive/MyDrive/AML"
ZIP_PATH      = os.path.join(CHALLENGE_DIR, "dataset.zip")
IMAGES_DIR    = "/content/images"   # extracted here (fast local SSD, not Drive)

if not os.path.exists(IMAGES_DIR):
    print("Extracting images...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(IMAGES_DIR)
    print(f"Done — extracted to {IMAGES_DIR}")
else:
    print(f"Already extracted: {IMAGES_DIR}")

# Peek at what's inside
sample_files = os.listdir(os.path.join(IMAGES_DIR,"images"))[:5]
print(f"Total images : {len(os.listdir(os.path.join(IMAGES_DIR,"images")))}")
print(f"Sample names : {sample_files}")
IMAGES_DIR = os.path.join(IMAGES_DIR,"images")

#### Load the Training Data

In [ ]:
import pandas as pd

train_df = pd.read_csv(os.path.join(CHALLENGE_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(CHALLENGE_DIR, "test.csv"))

# Map image_id → full path
def img_path(image_id):
    # Try common extensions; adjust if your images have a fixed extension
    for ext in ["", ".jpg", ".jpeg", ".png", ".webp"]:
        p = os.path.join(IMAGES_DIR, f"{image_id}{ext}")
        if os.path.exists(p):
            return p
    return None

train_df["path"] = train_df["image_id"].apply(img_path)
test_df["path"]  = test_df["image_id"].apply(img_path)

print(f"Train rows   : {len(train_df)}")
print(f"Test  rows   : {len(test_df)}")
print("\nLabel distribution:")
print(train_df["ground_truth"].value_counts().rename({0: "authentic", 1: "ai_generated"}))


## Explore data

### Feature Dimensionality

In [ ]:
!pip install umap-learn -q

#### Load features

In [ ]:
import numpy as np

FOLD_TO_RUN = 0

TRAIN_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_train_features.npy")
TRAIN_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_train_labels.npy")
VAL_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_val_features.npy")
VAL_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_val_labels.npy")
TEST_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_test_features.npy")
TEST_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_test_labels.npy")

X = np.vstack((np.load(TRAIN_FEATURES_PATH), np.load(VAL_FEATURES_PATH), np.load(TEST_FEATURES_PATH)))
y = np.concatenate((np.load(TRAIN_LABELS_PATH), np.load(VAL_LABELS_PATH), np.load(TEST_LABELS_PATH)))

model_name = "tf_efficientnetv2_s.in21k"

FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_extracted_features.npy")
LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_extracted_labels.npy")

Xb = np.load(FEATURES_PATH)
yb = np.load(LABELS_PATH)

#### Visualise

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import umap.umap_ as umap

plt.rcParams['font.size'] = 9

def visualize_comparison(X_frozen, y_frozen, X_fine, y_fine, filename="umap_comparison"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 5))

    label_map = {0: 'Authentic', 1: 'AI-Generated'}
    palette = {'Authentic': '#2171b5', 'AI-Generated': '#cb181d'}
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42)

    embedding_b = reducer.fit_transform(X_frozen)
    y_text_b = [label_map[label] for label in y_frozen]

    sns.scatterplot(
        x=embedding_b[:, 0], y=embedding_b[:, 1], hue=y_text_b,
        palette=palette, alpha=0.5, s=5, edgecolor=None, ax=ax1
    )
    ax1.set_title("(a) Frozen ImageNet-21k Features", fontsize=10, pad=10)
    ax1.set_xlabel('UMAP Dimension 1')
    ax1.set_ylabel('UMAP Dimension 2')
    ax1.get_legend().remove()

    embedding = reducer.fit_transform(X_fine)
    y_text = [label_map[label] for label in y_fine]

    sns.scatterplot(
        x=embedding[:, 0], y=embedding[:, 1], hue=y_text,
        palette=palette, alpha=0.5, s=5, edgecolor=None, ax=ax2
    )
    ax2.set_title("(b) Fine-Tuned Domain-Specific Features", fontsize=10, pad=10)
    ax2.set_xlabel('UMAP Dimension 1')
    ax2.set_ylabel('UMAP Dimension 2')

    handles, labels = ax2.get_legend_handles_labels()
    ax2.get_legend().remove()
    fig.legend(handles, labels, loc='lower center', ncol=2,
               bbox_to_anchor=(0.5, -0.05), fontsize='small')

    plt.tight_layout()
    plt.savefig(f'{filename}.pdf', format='pdf', dpi=300, bbox_inches='tight')
    plt.show()

visualize_comparison(Xb, yb, X, y)